# Feature Engineering and Data Cleaning
## Healthcare Provider Fraud Detection

This notebook focuses on data cleaning, handling missing values, creating new features, and preparing data for modeling.

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# Load cleaned data
benf = pd.read_csv('Train_Beneficiarydata-1542865627584.csv').drop_duplicates()
inpatient = pd.read_csv('Train_Inpatientdata-1542865627584.csv').drop_duplicates()
outpatient = pd.read_csv('Train_Outpatientdata-1542865627584.csv').drop_duplicates()
train_labels = pd.read_csv('Train-1542865627584.csv')

benf_test = pd.read_csv('Test_Beneficiarydata-1542969243754.csv').drop_duplicates()
inpatient_test = pd.read_csv('Test_Inpatientdata-1542969243754.csv').drop_duplicates()
outpatient_test = pd.read_csv('Test_Outpatientdata-1542969243754.csv').drop_duplicates()
test_id = pd.read_csv('Test-1542969243754.csv')

print("Data loaded successfully!")

## 2. Handle Missing Values

### 2.1 Missing Value Strategy

In [ ]:
def handle_missing_values(df, name, strategy='mean'):
    """
    Handle missing values in a dataframe
    
    Parameters:
    - df: dataframe
    - name: dataframe name for printing
    - strategy: 'mean', 'median', 'forward_fill', 'drop', 'zero'
    """
    df_copy = df.copy()
    missing_before = df_copy.isnull().sum().sum()
    
    numeric_cols = df_copy.select_dtypes(include=[np.number]).columns
    object_cols = df_copy.select_dtypes(include=['object']).columns
    
    # Handle numeric columns
    if strategy == 'mean':
        df_copy[numeric_cols] = df_copy[numeric_cols].fillna(df_copy[numeric_cols].mean())
    elif strategy == 'median':
        df_copy[numeric_cols] = df_copy[numeric_cols].fillna(df_copy[numeric_cols].median())
    elif strategy == 'zero':
        df_copy[numeric_cols] = df_copy[numeric_cols].fillna(0)
    
    # Handle object columns
    df_copy[object_cols] = df_copy[object_cols].fillna('Unknown')
    
    missing_after = df_copy.isnull().sum().sum()
    print(f"\n{name}:")
    print(f"  Missing values before: {missing_before}")
    print(f"  Missing values after: {missing_after}")
    
    return df_copy

# Apply to training data
benf = handle_missing_values(benf, "Beneficiary (Train)", strategy='mean')
inpatient = handle_missing_values(inpatient, "Inpatient (Train)", strategy='mean')
outpatient = handle_missing_values(outpatient, "Outpatient (Train)", strategy='mean')

In [ ]:
# Apply to test data
benf_test = handle_missing_values(benf_test, "Beneficiary (Test)", strategy='mean')
inpatient_test = handle_missing_values(inpatient_test, "Inpatient (Test)", strategy='mean')
outpatient_test = handle_missing_values(outpatient_test, "Outpatient (Test)", strategy='mean')

## 3. Feature Engineering

### 3.1 Inpatient Features

In [ ]:
def engineer_inpatient_features(df, name="Data"):
    """
    Engineer features from inpatient data
    """
    df_copy = df.copy()
    
    # Count features
    inpatient_claims = df_copy.groupby('BeneID').size().reset_index(name='InpatientClaimsCount')
    
    # Sum features
    inpatient_cost = df_copy.groupby('BeneID')[['ClaimAmount', 'DeductibleAmtPaid']].sum().reset_index()
    inpatient_cost.columns = ['BeneID', 'TotalInpatientCost', 'TotalInpatientDeductible']
    
    # Average features
    inpatient_avg = df_copy.groupby('BeneID')[['ClaimAmount']].mean().reset_index()
    inpatient_avg.columns = ['BeneID', 'AvgInpatientClaimAmount']
    
    # Merge all features
    inpatient_features = inpatient_claims.merge(inpatient_cost, on='BeneID')
    inpatient_features = inpatient_features.merge(inpatient_avg, on='BeneID')
    
    print(f"\n{name} - Inpatient Features Engineered:")
    print(f"  Created {len(inpatient_features.columns)-1} new features")
    print(f"  Features: {list(inpatient_features.columns[1:])}")
    
    return inpatient_features

inpatient_features = engineer_inpatient_features(inpatient, "Training")
inpatient_features_test = engineer_inpatient_features(inpatient_test, "Testing")

### 3.2 Outpatient Features

In [ ]:
def engineer_outpatient_features(df, name="Data"):
    """
    Engineer features from outpatient data
    """
    df_copy = df.copy()
    
    # Count features
    outpatient_claims = df_copy.groupby('BeneID').size().reset_index(name='OutpatientClaimsCount')
    
    # Sum features
    outpatient_cost = df_copy.groupby('BeneID')[['ClaimAmount', 'DeductibleAmtPaid']].sum().reset_index()
    outpatient_cost.columns = ['BeneID', 'TotalOutpatientCost', 'TotalOutpatientDeductible']
    
    # Average features
    outpatient_avg = df_copy.groupby('BeneID')[['ClaimAmount']].mean().reset_index()
    outpatient_avg.columns = ['BeneID', 'AvgOutpatientClaimAmount']
    
    # Merge all features
    outpatient_features = outpatient_claims.merge(outpatient_cost, on='BeneID')
    outpatient_features = outpatient_features.merge(outpatient_avg, on='BeneID')
    
    print(f"\n{name} - Outpatient Features Engineered:")
    print(f"  Created {len(outpatient_features.columns)-1} new features")
    print(f"  Features: {list(outpatient_features.columns[1:])}")
    
    return outpatient_features

outpatient_features = engineer_outpatient_features(outpatient, "Training")
outpatient_features_test = engineer_outpatient_features(outpatient_test, "Testing")

### 3.3 Beneficiary Features

In [ ]:
def engineer_beneficiary_features(df, name="Data"):
    """
    Engineer features from beneficiary data
    """
    df_copy = df.copy()
    
    # Calculate age (assuming current year is 2009)
    if 'DOB' in df_copy.columns:
        df_copy['Age'] = 2009 - pd.to_datetime(df_copy['DOB'], format='%Y%m%d').dt.year
    
    # Calculate financial ratios
    df_copy['IPReimbursement_to_Coverage'] = df_copy['IPAnnualReimbursementAmt'] / (df_copy['NoOfMonths_PartACovered'] + 1)
    df_copy['OPReimbursement_to_Coverage'] = df_copy['OPAnnualReimbursementAmt'] / (df_copy['NoOfMonths_PartBCovered'] + 1)
    df_copy['Total_IP_to_OP_Ratio'] = df_copy['IPAnnualReimbursementAmt'] / (df_copy['OPAnnualReimbursementAmt'] + 1)
    
    print(f"\n{name} - Beneficiary Features Engineered:")
    print(f"  Created ratio and age-based features")
    
    return df_copy

benf = engineer_beneficiary_features(benf, "Training")
benf_test = engineer_beneficiary_features(benf_test, "Testing")

## 4. Data Aggregation

In [ ]:
def aggregate_features(benf_df, inpatient_feat, outpatient_feat, labels_df=None, data_type="Train"):
    """
    Aggregate all features into a single dataset
    """
    # Merge inpatient and outpatient features
    combined = inpatient_feat.merge(outpatient_feat, on='BeneID', how='outer')
    combined = combined.fillna(0)
    
    # Merge with beneficiary data
    final_df = benf_df.merge(combined, left_on='BeneID', right_on='BeneID', how='left')
    final_df = final_df.fillna(0)
    
    # Add labels if provided
    if labels_df is not None:
        final_df = final_df.merge(labels_df, on='BeneID', how='left')
    
    print(f"\n{data_type} Data Aggregation Complete:")
    print(f"  Total records: {len(final_df)}")
    print(f"  Total features: {len(final_df.columns)}")
    print(f"  Shape: {final_df.shape}")
    
    return final_df

# Aggregate training data
X_train = aggregate_features(benf, inpatient_features, outpatient_features, train_labels, "Training")

# Aggregate test data
X_test = aggregate_features(benf_test, inpatient_features_test, outpatient_features_test, None, "Testing")

## 5. Data Preparation for Modeling

In [ ]:
# Separate features and target
if 'PotentialFraud' in X_train.columns:
    y_train = X_train['PotentialFraud']
    X_train = X_train.drop(['PotentialFraud', 'BeneID'], axis=1, errors='ignore')
    X_test = X_test.drop(['BeneID'], axis=1, errors='ignore')
    
    print(f"\nData Split:")
    print(f"  X_train shape: {X_train.shape}")
    print(f"  y_train shape: {y_train.shape}")
    print(f"  X_test shape: {X_test.shape}")
    print(f"\n  Target distribution:")
    print(f"    Non-Fraud: {(y_train == 0).sum()}")
    print(f"    Fraud: {(y_train == 1).sum()}")

## 6. Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Select numeric columns for scaling
numeric_features = X_train.select_dtypes(include=[np.number]).columns

print(f"\nNumeric features to scale: {len(numeric_features)}")

# Initialize scalers
scaler = StandardScaler()

# Fit on training data
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])

# Transform test data
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

print(f"  Features scaled using StandardScaler")
print(f"\n  X_train scaled stats:")
print(f"    Mean: {X_train[numeric_features].mean().mean():.4f}")
print(f"    Std: {X_train[numeric_features].std().mean():.4f}")

## 7. Save Processed Data

In [ ]:
# Save processed datasets
X_train.to_csv('X_train_processed.csv', index=False)
X_test.to_csv('X_test_processed.csv', index=False)
y_train.to_csv('y_train.csv', index=False)

print("\nProcessed data saved:")
print("  - X_train_processed.csv")
print("  - X_test_processed.csv")
print("  - y_train.csv")

## 8. Summary Statistics

In [ ]:
print("\n" + "="*60)
print("PROCESSED DATA SUMMARY")
print("="*60)
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nX_train info:")
print(X_train.info())
print(f"\nX_train describe:")
print(X_train.describe())